# PySpark Medallion — Full Analytics Dashboard
Data is read from Delta Lake **(Bronze → Silver → Gold)**.

## 1 · Setup

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("JAVA_HOME", "/usr/lib/jvm/java-17-openjdk-amd64")

# ── Project root ──────────────────────────────────────────────────────────────
# Notebooks live in <project>/notebooks/ so parent is project root.
PROJECT_ROOT = os.path.dirname(os.path.abspath(os.path.join(os.getcwd(), ".")))
# Fallback: if already at project root
if not os.path.isdir(os.path.join(PROJECT_ROOT, "config")):
    PROJECT_ROOT = os.getcwd()
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

# ── Python libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (14, 5),
                     "axes.titlesize": 13, "axes.labelsize": 11})
PALETTE = px.colors.qualitative.Plotly

# ── Spark + Analytics ─────────────────────────────────────────────────────────
from config.spark_config import SparkConfig
from analysis.business_analytics import BusinessAnalytics
from analysis.extended_analytics import ExtendedAnalytics

# Build one session and share it with every object.
# SparkConfig.get_spark_session() uses SparkSession.getActiveSession() first,
# so all objects below share the same session and the same view catalog.
spark = SparkConfig().get_spark_session()
spark.sparkContext.setLogLevel("ERROR")

ba = BusinessAnalytics(spark)
ea = ExtendedAnalytics(spark)

ba.register_views()   # daily_sales_summary, customer_analytics, ... silver_*
ea.register_views()   # g_daily_sales_summary, s_customers, ...

# Pre-fetch all analytics DataFrames now, while views are registered on this session.
sales = ba.run_sales_analytics()
cplx  = ba.run_complex_analytics()
adv   = ba.run_advanced_analytics()

print("\nSpark version:", spark.version)
print("All analytics ready ✅")

Project root: /home/siddi/spark-project


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/15 08:16:41 WARN Utils: Your hostname, siddi, resolves to a loopback address: 127.0.1.1; using 192.168.208.128 instead (on interface ens33)
26/07/15 08:16:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


26/07/15 08:16:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


✅ Registered gold view : daily_sales_summary


...

✅ Registered silver view: s_stores
✅ Registered silver view: s_orders


✅ Registered silver view: s_sales
✅ Registered gold view  : g_daily_sales_summary


✅ Registered gold view  : g_customer_analytics
✅ Registered gold view  : g_product_performance


✅ Registered gold view  : g_store_performance
✅ Registered gold view  : g_monthly_time_series



Spark version: 4.1.2
All analytics ready ✅


## 2 · Monthly Revenue by Category

In [2]:
monthly_pdf = sales["monthly_revenue"].toPandas()
monthly_pdf["period"] = (monthly_pdf["year"].astype(str) + "-" +
                         monthly_pdf["month"].astype(str).str.zfill(2))
for c in ["monthly_revenue"]:
    monthly_pdf[c] = pd.to_numeric(monthly_pdf[c], errors="coerce").astype(float)

fig = px.bar(monthly_pdf, x="period", y="monthly_revenue", color="category",
             title="Monthly Revenue by Category",
             labels={"monthly_revenue": "Revenue (£)", "period": "Month", "category": "Category"},
             color_discrete_sequence=PALETTE)
fig.update_layout(xaxis_tickangle=-45, hovermode="x unified")
fig.show()

In [3]:
pivot = monthly_pdf.pivot_table(index="period", columns="category",
                                values="monthly_revenue", aggfunc="sum").fillna(0)
fig2, ax = plt.subplots(figsize=(16, 5))
pivot.plot.area(ax=ax, alpha=0.75, colormap="tab10")
ax.set_title("Monthly Revenue — Stacked Area by Category")
ax.set_xlabel("Month"); ax.set_ylabel("Revenue (£)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
ax.legend(loc="upper left", fontsize=9)
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

## 3 · Revenue Trend & Moving Averages

In [4]:
ma_pdf = adv["moving_averages"].toPandas()
ma_pdf["sale_date"] = pd.to_datetime(ma_pdf["sale_date"])
for c in ["daily_revenue", "moving_avg_7days", "moving_avg_30days"]:
    ma_pdf[c] = pd.to_numeric(ma_pdf[c], errors="coerce").astype(float)
ma_pdf.sort_values("sale_date", inplace=True)

fig = go.Figure([
    go.Scatter(x=ma_pdf["sale_date"], y=ma_pdf["daily_revenue"],
               mode="lines", name="Daily Revenue",
               line=dict(color="lightblue", width=1), opacity=0.6),
    go.Scatter(x=ma_pdf["sale_date"], y=ma_pdf["moving_avg_7days"],
               mode="lines", name="7-Day MA", line=dict(color="steelblue", width=2)),
    go.Scatter(x=ma_pdf["sale_date"], y=ma_pdf["moving_avg_30days"],
               mode="lines", name="30-Day MA", line=dict(color="darkred", width=2.5)),
])
fig.update_layout(title="Daily Revenue with Moving Averages",
                  xaxis_title="Date", yaxis_title="Revenue (£)",
                  hovermode="x unified")
fig.show()

## 4 · Year-over-Year Revenue Comparison

In [5]:
yoy_pdf = adv["yoy_comparison"].toPandas()
yoy_pdf["yoy_growth_pct"] = pd.to_numeric(yoy_pdf["yoy_growth_pct"], errors="coerce").astype(float)
yoy_pdf["period"] = (yoy_pdf["year"].astype(str) + "-" +
                     yoy_pdf["month"].astype(str).str.zfill(2))

cats = sorted(yoy_pdf["category"].dropna().unique())
fig, axes = plt.subplots(1, len(cats), figsize=(max(16, 3 * len(cats)), 5), sharey=False)
if len(cats) == 1:
    axes = [axes]
for ax, cat in zip(axes, cats):
    sub = yoy_pdf[yoy_pdf["category"] == cat].dropna(subset=["yoy_growth_pct"])
    bar_colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in sub["yoy_growth_pct"]]
    ax.bar(sub["period"], sub["yoy_growth_pct"], color=bar_colors)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(cat, fontsize=10)
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    ax.set_ylabel("YoY Growth %")
plt.suptitle("Year-over-Year Revenue Growth by Category", fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

## 5 · Top 10 Customers by Lifetime Value

In [6]:
top_cust = sales["top_customers"].toPandas()
for c in ["lifetime_value", "total_orders", "avg_order_value"]:
    top_cust[c] = pd.to_numeric(top_cust[c], errors="coerce").astype(float)
top_cust["full_name"] = top_cust["first_name"] + " " + top_cust["last_name"]

fig = px.bar(top_cust.sort_values("lifetime_value"),
             x="lifetime_value", y="full_name", orientation="h",
             color="customer_segment", text="lifetime_value",
             title="Top 10 Customers by Lifetime Value",
             labels={"lifetime_value": "LTV (£)", "full_name": "Customer",
                     "customer_segment": "Segment"},
             color_discrete_sequence=PALETTE)
fig.update_traces(texttemplate="£%{text:,.0f}", textposition="outside")
fig.update_layout(xaxis_tickformat="£,.0f", height=420)
fig.show()

In [7]:
fig2 = px.scatter(top_cust,
                  x="total_orders", y="lifetime_value",
                  size="avg_order_value", color="churn_risk",
                  hover_name="full_name",
                  title="Top Customers — Orders vs LTV  (bubble = avg order size)",
                  labels={"total_orders": "Total Orders", "lifetime_value": "LTV (£)",
                          "churn_risk": "Churn Risk"},
                  color_discrete_map={"Low": "#2ecc71", "Medium": "#f39c12", "High": "#e74c3c"})
fig2.update_yaxes(tickformat="£,.0f")
fig2.show()

## 6 · Customer Lifetime Value Distribution

In [8]:
ltv_raw = spark.read.format("delta").load(
    os.path.join(PROJECT_ROOT, "data", "gold", "customer_analytics")
).select("lifetime_value").toPandas()
ltv_raw["lifetime_value"] = pd.to_numeric(ltv_raw["lifetime_value"], errors="coerce").astype(float)
vals = ltv_raw["lifetime_value"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].hist(vals, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
for p, col in zip([25, 50, 75], ["#f39c12", "red", "#9b59b6"]):
    v = vals.quantile(p / 100)
    axes[0].axvline(v, linestyle="--", color=col, alpha=0.8, label=f"P{p}: £{v:,.0f}")
axes[0].set_title("LTV Histogram"); axes[0].set_xlabel("LTV (£)")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
axes[0].legend(fontsize=9)
axes[1].boxplot(vals, vert=False, patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.5))
axes[1].set_title("LTV Box Plot"); axes[1].set_xlabel("LTV (£)")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
plt.suptitle("Customer Lifetime Value Distribution", fontsize=13)
plt.tight_layout(); plt.show()

pcts = [10, 25, 50, 75, 90, 95, 99]
pd.DataFrame({"Percentile": [f"P{p}" for p in pcts],
              "LTV": [f"£{vals.quantile(p/100):,.2f}" for p in pcts]
             }).set_index("Percentile")

,LTV
Percentile,
P10,£494.18
P25,"£1,530.65"
P50,"£3,646.37"
P75,"£6,847.71"
P90,"£10,179.80"
P95,"£12,021.41"
P99,"£18,268.71"


## 7 · Customer Segment & Churn Risk

In [9]:
ca_pdf = spark.read.format("delta").load(
    os.path.join(PROJECT_ROOT, "data", "gold", "customer_analytics")
).toPandas()
for c in ["lifetime_value", "total_orders", "avg_order_value"]:
    ca_pdf[c] = pd.to_numeric(ca_pdf[c], errors="coerce").astype(float)

seg = ca_pdf.groupby("customer_segment").agg(
    customers=("customer_id", "count"),
    avg_ltv=("lifetime_value", "mean"),
    avg_orders=("total_orders", "mean"),
).reset_index()

fig = make_subplots(1, 3, subplot_titles=(
    "Customers per Segment", "Avg LTV per Segment", "Avg Orders per Segment"))
for idx, col in enumerate(["customers", "avg_ltv", "avg_orders"], 1):
    fig.add_trace(go.Bar(x=seg["customer_segment"], y=seg[col].round(2),
                         marker_color=PALETTE[:len(seg)], showlegend=False),
                  row=1, col=idx)
fig.update_layout(title_text="Customer Segment Overview", height=380)
fig.show()

churn = ca_pdf["churn_risk"].value_counts().reset_index()
churn.columns = ["churn_risk", "count"]
fig2 = px.pie(churn, values="count", names="churn_risk", title="Churn Risk Distribution",
              hole=0.4, color="churn_risk",
              color_discrete_map={"Low": "#2ecc71", "Medium": "#f39c12", "High": "#e74c3c"})
fig2.update_traces(textinfo="percent+label")
fig2.show()

## 8 · RFM Segmentation

In [10]:
rfm_pdf = cplx["rfm_analysis"].toPandas()
for c in ["recency", "frequency", "monetary"]:
    rfm_pdf[c] = pd.to_numeric(rfm_pdf[c], errors="coerce").astype(float)

seg_sum = (rfm_pdf.groupby("rfm_segment")
           .agg(customers=("customer_id", "count"),
                avg_recency=("recency", "mean"),
                avg_monetary=("monetary", "mean"))
           .reset_index().sort_values("avg_monetary", ascending=False))

fig = px.bar(seg_sum, x="rfm_segment", y="customers",
             color="avg_monetary", color_continuous_scale="Blues",
             title="RFM Segments — Customer Count  (colour = avg revenue)",
             labels={"rfm_segment": "Segment", "customers": "Customers",
                     "avg_monetary": "Avg Revenue (£)"})
fig.update_layout(xaxis_tickangle=-30)
fig.show()

In [11]:
sample = rfm_pdf.sample(min(300, len(rfm_pdf)), random_state=42)
fig2 = px.scatter(sample, x="recency", y="frequency", size="monetary",
                  color="rfm_segment", hover_name="first_name",
                  title="RFM Bubble Chart — Recency vs Frequency  (bubble = monetary)",
                  labels={"recency": "Recency (days)", "frequency": "Orders",
                          "rfm_segment": "Segment"},
                  size_max=30, color_discrete_sequence=PALETTE)
fig2.show()

In [12]:
ext_rfm = ea.rfm_segmentation().toPandas()
for c in ["recency_days", "frequency", "monetary"]:
    ext_rfm[c] = pd.to_numeric(ext_rfm[c], errors="coerce").astype(float)
ext_seg = (ext_rfm.groupby("segment")
           .agg(customers=("customer_id", "count"), avg_revenue=("monetary", "mean"))
           .reset_index().sort_values("avg_revenue", ascending=False))
fig3 = px.funnel(ext_seg.sort_values("customers", ascending=False),
                 x="customers", y="segment", color="avg_revenue",
                 title="Extended RFM — Customer Funnel by Segment")
fig3.show()


━━━ 2. RFM — Champions ━━━


+-----------+-----------+---------+---------------+--------------------------------------------+------------+---------+--------+-------+-------+-------+---------+--------+---------+
|customer_id|first_name |last_name|city           |country                                     |recency_days|frequency|monetary|r_score|f_score|m_score|rfm_total|rfm_code|segment  |
+-----------+-----------+---------+---------------+--------------------------------------------+------------+---------+--------+-------+-------+-------+---------+--------+---------+
|881        |LAURA      |LEE      |Robertchester  |Pitcairn Islands                            |132         |2        |9337.56 |5      |5      |5      |15       |555     |Champions|
|822        |JUDITH     |ROGERS   |East Amber     |Heard Island And Mcdonald Islands           |64          |2        |7602.28 |5      |5      |5      |15       |555     |Champions|
|222        |CHRISTOPHER|STEWART  |New Jocelyn    |Chile                                  


  RFM Segment Summary:


+-------------------+---------+-----------+----------------+
|segment            |customers|avg_revenue|avg_recency_days|
+-------------------+---------+-----------+----------------+
|At Risk            |16       |9644.97    |503.4           |
|Champions          |39       |8957.76    |83.2            |
|Loyal Customers    |82       |6774.46    |168.2           |
|Potential Loyalists|73       |5956.31    |312.5           |
|Promising          |11       |5682.34    |382.1           |
|Hibernating        |92       |5375.02    |573.3           |
|Cant Lose Them     |3        |2687.14    |555.3           |
|Lost               |169      |1263.50    |376.9           |
+-------------------+---------+-----------+----------------+



## 9 · Cohort Retention Matrix

In [13]:
cohort_pdf = ea.cohort_retention_matrix().toPandas()
cohort_pdf["cohort_month"] = pd.to_datetime(cohort_pdf["cohort_month"]).dt.strftime("%Y-%m")
for c in ["months_since_first", "retention_pct"]:
    cohort_pdf[c] = pd.to_numeric(cohort_pdf[c], errors="coerce").astype(float)

pivot = cohort_pdf.pivot_table(index="cohort_month", columns="months_since_first",
                               values="retention_pct", aggfunc="mean")
fig, ax = plt.subplots(figsize=(18, max(6, len(pivot) * 0.45)))
sns.heatmap(pivot, annot=True, fmt=".0f", linewidths=0.5,
            cmap="YlOrRd_r", ax=ax, annot_kws={"size": 9},
            cbar_kws={"label": "Retention %"})
ax.set_title("Monthly Cohort Retention Matrix (%)", fontsize=14, pad=12)
ax.set_xlabel("Months Since First Purchase"); ax.set_ylabel("Cohort Month")
plt.tight_layout(); plt.show()


━━━ 1. Cohort Retention Matrix (first 20 rows) ━━━


+-------------------+-----------+------------------+----------------+-------------+
|cohort_month       |cohort_size|months_since_first|active_customers|retention_pct|
+-------------------+-----------+------------------+----------------+-------------+
|2024-07-01 00:00:00|27         |0                 |27              |100.0        |
|2024-07-01 00:00:00|27         |1                 |1               |3.7          |
|2024-07-01 00:00:00|27         |3                 |2               |7.4          |
|2024-07-01 00:00:00|27         |4                 |6               |22.2         |
|2024-07-01 00:00:00|27         |6                 |2               |7.4          |
|2024-07-01 00:00:00|27         |7                 |3               |11.1         |
|2024-07-01 00:00:00|27         |9                 |1               |3.7          |
|2024-07-01 00:00:00|27         |10                |1               |3.7          |
|2024-07-01 00:00:00|27         |11                |1               |3.7    

## 10 · Product Performance Ranking

In [14]:
prod_pdf = sales["product_ranking"].toPandas()
for c in ["total_revenue", "profit_margin", "total_units_sold"]:
    prod_pdf[c] = pd.to_numeric(prod_pdf[c], errors="coerce").astype(float)

top20 = prod_pdf.head(20).sort_values("total_revenue")
fig = px.bar(top20, x="total_revenue", y="product_name", orientation="h",
             color="product_category", text="total_revenue",
             title="Top 20 Products by Revenue",
             labels={"total_revenue": "Revenue (£)", "product_name": "Product",
                     "product_category": "Quadrant"},
             color_discrete_map={"Star": "#f1c40f", "Cash Cow": "#2ecc71",
                                  "High Profit": "#3498db", "Average": "#bdc3c7"})
fig.update_traces(texttemplate="£%{text:,.0f}", textposition="outside")
fig.update_layout(xaxis_tickformat="£,.0f", height=600)
fig.show()

In [15]:
fig2 = px.scatter(prod_pdf.head(50),
                  x="total_revenue", y="profit_margin",
                  size="total_units_sold", color="category",
                  hover_name="product_name",
                  title="Revenue vs Profit Margin — top 50  (bubble = units sold)",
                  labels={"total_revenue": "Revenue (£)", "profit_margin": "Margin %"},
                  size_max=35, color_discrete_sequence=PALETTE)
fig2.add_hline(y=float(prod_pdf["profit_margin"].median()),
               line_dash="dash", line_color="red",
               annotation_text=f"Median {prod_pdf['profit_margin'].median():.1f}%")
fig2.update_xaxes(tickformat="£,.0f")
fig2.show()

## 11 · ABC Classification & Pareto Analysis

In [16]:
abc_pdf = ea.abc_product_classification().toPandas()
for c in ["revenue", "cumulative_pct", "profit_margin"]:
    abc_pdf[c] = pd.to_numeric(abc_pdf[c], errors="coerce").astype(float)

abc_colors = {"A": "#2ecc71", "B": "#f39c12", "C": "#e74c3c"}
abc_sum = (abc_pdf.groupby("abc_class")
           .agg(products=("product_name", "count"),
                total_revenue=("revenue", "sum"),
                avg_margin=("profit_margin", "mean"))
           .reset_index())
c_list = [abc_colors.get(c, "grey") for c in abc_sum["abc_class"]]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].bar(abc_sum["abc_class"], abc_sum["products"], color=c_list)
axes[0].set_title("Products per Class"); axes[0].set_ylabel("Count")
axes[1].bar(abc_sum["abc_class"], abc_sum["total_revenue"], color=c_list)
axes[1].set_title("Revenue per Class"); axes[1].set_ylabel("(£)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
axes[2].bar(abc_sum["abc_class"], abc_sum["avg_margin"], color=c_list)
axes[2].set_title("Avg Margin per Class"); axes[2].set_ylabel("%")
plt.suptitle("ABC Product Classification", fontsize=13)
plt.tight_layout(); plt.show()


━━━ 3. ABC Product Classification ━━━


+---------+-------------+-------------+----------+
|abc_class|product_count|total_revenue|avg_margin|
+---------+-------------+-------------+----------+
|A        |278          |1817772.35   |52.73     |
|B        |161          |340300.20    |50.32     |
|C        |177          |114516.76    |50.01     |
+---------+-------------+-------------+----------+



In [17]:
pareto_pdf = adv["pareto_analysis"].toPandas()
for c in ["cumulative_percentage", "total_sales"]:
    pareto_pdf[c] = pd.to_numeric(pareto_pdf[c], errors="coerce").astype(float)
pareto_pdf = pareto_pdf.sort_values("rank")

fig2, ax1 = plt.subplots(figsize=(14, 5))
bar_cols = [abc_colors.get(c, "grey") for c in pareto_pdf["pareto_category"]]
ax1.bar(pareto_pdf["rank"], pareto_pdf["total_sales"], color=bar_cols, alpha=0.8)
ax1.set_xlabel("Product Rank"); ax1.set_ylabel("Sales (£)")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
ax2 = ax1.twinx()
ax2.plot(pareto_pdf["rank"], pareto_pdf["cumulative_percentage"],
         color="darkred", linewidth=2, label="Cumulative %")
ax2.axhline(80, color="red", linestyle="--", alpha=0.6, label="80% threshold")
ax2.set_ylabel("Cumulative %"); ax2.set_ylim(0, 110)
ax2.legend(loc="center right")
ax1.set_title("Pareto Chart — Product Sales Distribution")
plt.tight_layout(); plt.show()

## 12 · Store Performance

In [18]:
store_pdf = sales["store_comparison"].toPandas()
for c in ["total_revenue", "profit_margin", "unique_customers", "avg_order_value"]:
    store_pdf[c] = pd.to_numeric(store_pdf[c], errors="coerce").astype(float)

fig = px.bar(store_pdf.head(15).sort_values("total_revenue"),
             x="total_revenue", y="store_name", orientation="h",
             color="profit_margin", color_continuous_scale="Greens",
             text="total_revenue",
             title="Top 15 Stores by Revenue  (colour = margin %)",
             labels={"total_revenue": "Revenue (£)", "store_name": "Store"})
fig.update_traces(texttemplate="£%{text:,.0f}", textposition="outside")
fig.update_layout(height=520, xaxis_tickformat="£,.0f")
fig.show()

In [19]:
fig2 = px.scatter(store_pdf.dropna(subset=["total_revenue", "unique_customers"]),
                  x="unique_customers", y="total_revenue",
                  size="avg_order_value", color="region",
                  hover_name="store_name", hover_data=["city", "country"],
                  title="Stores — Unique Customers vs Revenue  (bubble = avg order value)",
                  labels={"unique_customers": "Unique Customers",
                          "total_revenue": "Revenue (£)", "region": "Region"},
                  size_max=30, color_discrete_sequence=PALETTE)
fig2.update_yaxes(tickformat="£,.0f")
fig2.show()

## 13 · Category × Country Revenue Heatmap

In [20]:
heatmap_pdf = ea.category_store_heatmap().toPandas()
heatmap_pdf["revenue"] = pd.to_numeric(heatmap_pdf["revenue"], errors="coerce").astype(float)

top_countries = (heatmap_pdf.groupby("store_country")["revenue"]
                 .sum().nlargest(20).index.tolist())
hm = heatmap_pdf[heatmap_pdf["store_country"].isin(top_countries)]
pivot_h = hm.pivot_table(index="category", columns="store_country",
                         values="revenue", aggfunc="sum").fillna(0)

fig, ax = plt.subplots(figsize=(22, 6))
sns.heatmap(pivot_h, annot=True, fmt=",.0f", linewidths=0.4,
            cmap="YlGnBu", ax=ax, annot_kws={"size": 8},
            cbar_kws={"label": "Revenue (£)"})
ax.set_title("Revenue Heatmap — Category × Country (Top 20 Countries)", fontsize=13)
ax.set_xlabel("Country"); ax.set_ylabel("Category")
plt.tight_layout(); plt.show()


━━━ 7. Category × Country Heatmap (top 20) ━━━


+-----------+--------------------------------+--------+-----+----------+
|category   |store_country                   |revenue |units|avg_margin|
+-----------+--------------------------------+--------+-----+----------+
|Apparel    |Sri Lanka                       |20248.56|29   |58.84     |
|Sports     |Saint Vincent And The Grenadines|19324.16|26   |58.63     |
|Electronics|Vietnam                         |18609.89|38   |62.89     |
|Books      |Norfolk Island                  |18033.78|27   |48.23     |
|Electronics|Niger                           |17487.54|23   |56.02     |
|Electronics|Puerto Rico                     |16393.54|29   |60.16     |
|Furniture  |Grenada                         |16079.49|28   |64.34     |
|Electronics|Afghanistan                     |14950.12|16   |63.39     |
|Sports     |Poland                          |14609.33|26   |52.98     |
|Food       |Ukraine                         |14136.08|26   |57.92     |
|Books      |Azerbaijan                      |14035

## 14 · Basket / Cross-Sell Analysis

In [21]:
basket_pdf = ea.basket_analysis().toPandas()
basket_pdf["order_count"] = pd.to_numeric(basket_pdf["order_count"], errors="coerce").astype(float)
basket_pdf["combo"] = basket_pdf["cats"].apply(
    lambda x: " + ".join(sorted(set(x))) if isinstance(x, list) else str(x))
top_combos = basket_pdf.groupby("combo")["order_count"].sum().nlargest(15).reset_index()

fig = px.bar(top_combos.sort_values("order_count"),
             x="order_count", y="combo", orientation="h",
             title="Top 15 Category Co-Purchase Combinations",
             labels={"order_count": "Order Count", "combo": "Category Combo"},
             color="order_count", color_continuous_scale="Oranges")
fig.update_layout(height=480, showlegend=False)
fig.show()


━━━ 4. Basket Analysis — Top 20 Category Combos ━━━


+------------------------+-------------------+-----------+
|cats                    |distinct_categories|order_count|
+------------------------+-------------------+-----------+
|[Food, Sports]          |2                  |17         |
|[Electronics]           |1                  |12         |
|[Electronics, Apparel]  |2                  |11         |
|[Sports, Food]          |2                  |10         |
|[Furniture, Books]      |2                  |10         |
|[Food, Apparel]         |2                  |9          |
|[Food, Electronics]     |2                  |9          |
|[Food, Books]           |2                  |9          |
|[Books, Food]           |2                  |8          |
|[Electronics, Books]    |2                  |8          |
|[Furniture, Sports]     |2                  |7          |
|[Sports, Apparel]       |2                  |7          |
|[Furniture, Electronics]|2                  |7          |
|[Apparel]               |1                  |7         

In [22]:
cross_pdf = cplx["cross_sell_analysis"].toPandas()
cross_pdf["product_count"] = pd.to_numeric(cross_pdf["product_count"], errors="coerce").astype(float)
max_prod = int(cross_pdf["product_count"].dropna().max()) if not cross_pdf["product_count"].isna().all() else 5
fig2, ax = plt.subplots(figsize=(10, 4))
ax.hist(cross_pdf["product_count"].dropna(), bins=range(2, max_prod + 2),
        color="coral", edgecolor="white", alpha=0.85)
ax.set_title("Distribution of Products per Multi-Item Order")
ax.set_xlabel("Products in Order"); ax.set_ylabel("Number of Orders")
plt.tight_layout(); plt.show()

## 15 · Discount Effectiveness

In [23]:
disc_pdf = ea.discount_effectiveness().toPandas()
for c in ["avg_qty", "avg_order_value", "avg_profit", "overall_margin_pct", "sales_count"]:
    disc_pdf[c] = pd.to_numeric(disc_pdf[c], errors="coerce").astype(float)

bucket_order = ["0%", "1-10%", "11-20%", "21-30%", ">30%"]
disc_pdf["discount_bucket"] = pd.Categorical(disc_pdf["discount_bucket"],
                                              categories=bucket_order, ordered=True)
disc_pdf = disc_pdf.sort_values("discount_bucket")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, title, cmap in zip(
    axes,
    ["avg_order_value", "avg_profit", "overall_margin_pct"],
    ["Avg Order Value (£)", "Avg Profit (£)", "Overall Margin %"],
    ["Blues_d", "Greens_d", "Reds_d"]
):
    ax.bar(disc_pdf["discount_bucket"], disc_pdf[col],
           color=sns.color_palette(cmap, len(disc_pdf)))
    ax.set_title(title)
    if "£" in title:
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
plt.suptitle("Discount Effectiveness Analysis", fontsize=13)
plt.tight_layout(); plt.show()


━━━ 8. Discount Effectiveness ━━━


+---------------+-----------+-------+---------------+----------+------------------+
|discount_bucket|sales_count|avg_qty|avg_order_value|avg_profit|overall_margin_pct|
+---------------+-----------+-------+---------------+----------+------------------+
|0%             |44         |4.82   |2164.15        |1070.67   |49.47             |
|1-10%          |466        |5.43   |2421.20        |1220.60   |50.41             |
|11-20%         |490        |5.39   |2140.99        |1200.24   |56.06             |
+---------------+-----------+-------+---------------+----------+------------------+



## 16 · Day-of-Week & Monthly Seasonality

In [24]:
dow_pdf, _ = ea.seasonality_analysis()
dow_pdf = dow_pdf.toPandas()
for c in ["avg_daily_revenue", "total_revenue", "avg_units"]:
    dow_pdf[c] = pd.to_numeric(dow_pdf[c], errors="coerce").astype(float)

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_pdf["weekday"] = pd.Categorical(dow_pdf["weekday"], categories=day_order, ordered=True)
dow_pdf = dow_pdf.sort_values("weekday")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
bar_colors = sns.color_palette("coolwarm", len(dow_pdf))
axes[0].bar(dow_pdf["weekday"], dow_pdf["avg_daily_revenue"], color=bar_colors)
axes[0].set_title("Avg Daily Revenue by Day of Week"); axes[0].set_ylabel("Avg Revenue (£)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
axes[0].tick_params(axis="x", rotation=30)
axes[1].bar(dow_pdf["weekday"], dow_pdf["avg_units"], color=bar_colors)
axes[1].set_title("Avg Units Sold by Day of Week"); axes[1].set_ylabel("Avg Units")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()


━━━ 9. Day-of-Week Seasonality ━━━


+---------+-----------+-----------------+-------------+---------+
|weekday  |data_points|avg_daily_revenue|total_revenue|avg_units|
+---------+-----------+-----------------+-------------+---------+
|Monday   |142        |2208.08          |313547.00    |5.88     |
|Tuesday  |121        |2156.60          |260948.21    |5.46     |
|Wednesday|137        |2426.81          |332473.21    |5.7      |
|Thursday |130        |2693.25          |350122.34    |5.93     |
|Friday   |158        |2767.83          |437316.50    |6.08     |
|Saturday |133        |2222.73          |295622.68    |5.47     |
|Sunday   |108        |2616.29          |282559.37    |6.0      |
+---------+-----------+-----------------+-------------+---------+

  Monthly Revenue Trend:


+----+-----+---------+-----+
|year|month|revenue  |units|
+----+-----+---------+-----+
|2024|7    |91432.86 |228  |
|2024|8    |111921.95|253  |
|2024|9    |87692.22 |199  |
|2024|10   |94010.50 |242  |
|2024|11   |114987.27|293  |
|2024|12   |152852.42|363  |
|2025|1    |76264.55 |186  |
|2025|2    |53758.54 |162  |
|2025|3    |85228.37 |209  |
|2025|4    |93540.85 |204  |
|2025|5    |122042.52|236  |
|2025|6    |60811.30 |175  |
|2025|7    |117153.54|270  |
|2025|8    |112663.78|242  |
|2025|9    |56602.70 |143  |
|2025|10   |100588.99|266  |
|2025|11   |152745.22|290  |
|2025|12   |134915.09|335  |
|2026|1    |84954.64 |222  |
|2026|2    |59045.39 |117  |
|2026|3    |99019.28 |233  |
|2026|4    |86084.02 |190  |
|2026|5    |52593.13 |117  |
|2026|6    |60075.32 |170  |
+----+-----+---------+-----+
only showing top 24 rows


In [25]:
# Monthly revenue heatmap using g_monthly_time_series (registered by ea.register_views)
monthly_raw = spark.sql('''
    SELECT year, month, ROUND(SUM(monthly_revenue), 2) AS revenue
    FROM g_monthly_time_series GROUP BY year, month ORDER BY year, month
''').toPandas()
monthly_raw["revenue"] = pd.to_numeric(monthly_raw["revenue"], errors="coerce").astype(float)
monthly_raw["year"]    = monthly_raw["year"].astype(int)
monthly_raw["month"]   = monthly_raw["month"].astype(int)

pivot_m = monthly_raw.pivot_table(index="year", columns="month",
                                   values="revenue", aggfunc="sum").fillna(0)
pivot_m.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][:len(pivot_m.columns)]

fig2, ax = plt.subplots(figsize=(14, max(3, len(pivot_m))))
sns.heatmap(pivot_m, annot=True, fmt=",.0f", cmap="YlOrRd",
            linewidths=0.5, ax=ax, annot_kws={"size": 9},
            cbar_kws={"label": "Revenue (£)"})
ax.set_title("Monthly Revenue Heatmap  (Year × Month)", fontsize=13)
plt.tight_layout(); plt.show()

## 17 · Sales Anomaly Detection

In [26]:
anom_pdf = ea.anomaly_detection().toPandas()
anom_pdf["sale_date"] = pd.to_datetime(anom_pdf["sale_date"])
for c in ["revenue", "z_score"]:
    anom_pdf[c] = pd.to_numeric(anom_pdf[c], errors="coerce").astype(float)

# Full daily series via g_daily_sales_summary (registered by ea)
daily_rev = spark.sql('''
    SELECT sale_date, category, ROUND(SUM(total_revenue), 2) AS revenue
    FROM g_daily_sales_summary GROUP BY sale_date, category ORDER BY sale_date
''').toPandas()
daily_rev["sale_date"] = pd.to_datetime(daily_rev["sale_date"])
daily_rev["revenue"]   = pd.to_numeric(daily_rev["revenue"], errors="coerce").astype(float)

cats = sorted(daily_rev["category"].dropna().unique())
fig, axes = plt.subplots(len(cats), 1, figsize=(16, 3.5 * len(cats)), sharex=True)
if len(cats) == 1:
    axes = [axes]
for ax, cat in zip(axes, cats):
    sub  = daily_rev[daily_rev["category"] == cat].sort_values("sale_date")
    anom = anom_pdf[anom_pdf["category"] == cat]
    ax.plot(sub["sale_date"], sub["revenue"], color="steelblue", linewidth=1, alpha=0.7)
    if len(anom):
        ax.scatter(anom["sale_date"], anom["revenue"], color="red", zorder=5, s=55,
                   label=f"{len(anom)} anomalies")
        ax.legend(fontsize=8)
    ax.set_title(cat, fontsize=11); ax.set_ylabel("Revenue (£)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
axes[-1].set_xlabel("Date")
plt.suptitle("Daily Revenue with Anomalies (|z| > 2.5 per category)", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


━━━ 10. Sales Anomalies (|z| > 2.5) ━━━


+----------+-----------+--------+-------+-------+
|sale_date |category   |revenue |z_score|flag   |
+----------+-----------+--------+-------+-------+
|2024-08-09|Apparel    |14521.09|5.564  |ANOMALY|
|2024-07-31|Electronics|13417.33|4.554  |ANOMALY|
|2025-08-29|Furniture  |11548.26|4.332  |ANOMALY|
|2025-05-16|Sports     |10716.23|3.876  |ANOMALY|
|2024-11-28|Food       |9963.09 |3.837  |ANOMALY|
|2025-10-19|Books      |11464.64|3.763  |ANOMALY|
|2025-07-25|Sports     |9892.87 |3.492  |ANOMALY|
|2025-07-28|Sports     |9620.72 |3.365  |ANOMALY|
|2024-08-06|Sports     |9463.86 |3.292  |ANOMALY|
|2025-04-10|Electronics|9683.10 |2.973  |ANOMALY|
|2025-05-01|Food       |8149.78 |2.939  |ANOMALY|
|2024-07-23|Books      |9044.01 |2.727  |ANOMALY|
|2024-09-24|Electronics|8928.26 |2.653  |ANOMALY|
|2025-01-29|Sports     |8060.54 |2.637  |ANOMALY|
|2024-10-30|Apparel    |8115.93 |2.626  |ANOMALY|
|2025-08-29|Apparel    |8019.75 |2.582  |ANOMALY|
|2025-03-05|Books      |8592.96 |2.534  |ANOMALY|


## 18 · Employee Performance

In [27]:
emp_pdf = ea.employee_performance().toPandas()
for c in ["revenue_generated", "total_profit", "profit_margin_pct",
          "revenue_to_salary_ratio", "salary", "orders_handled"]:
    emp_pdf[c] = pd.to_numeric(emp_pdf[c], errors="coerce").astype(float)

fig = px.bar(emp_pdf.head(20).sort_values("revenue_generated"),
             x="revenue_generated", y="employee_name", orientation="h",
             color="department", text="revenue_generated",
             title="Top 20 Employees by Revenue Generated",
             labels={"revenue_generated": "Revenue (£)", "employee_name": "Employee",
                     "department": "Department"},
             color_discrete_sequence=PALETTE)
fig.update_traces(texttemplate="£%{text:,.0f}", textposition="outside")
fig.update_layout(height=580, xaxis_tickformat="£,.0f")
fig.show()


━━━ 11. Top 10 Employees by Revenue ━━━


+-----------+-------------------+----------+-------------------------+--------------+-----------------+---------------+------------+-----------------+-----------------------+------------+
|employee_id|employee_name      |department|salary                   |orders_handled|revenue_generated|avg_order_value|total_profit|profit_margin_pct|revenue_to_salary_ratio|revenue_rank|
+-----------+-------------------+----------+-------------------------+--------------+-----------------+---------------+------------+-----------------+-----------------------+------------+
|401        |Christopher Johnson|Finance   |58991.000000000000000000 |3             |23027.84         |3837.97        |12066.95    |52.40            |0.3904                 |1           |
|746        |Jessica Wilson     |HR        |65492.000000000000000000 |2             |22690.94         |3781.82        |12612.32    |55.58            |0.3465                 |2           |
|149        |Darlene Parker     |IT        |70855.0000000000

In [28]:
dept = (emp_pdf.groupby("department")
        .agg(employees=("employee_id", "count"),
             avg_revenue=("revenue_generated", "mean"),
             avg_margin=("profit_margin_pct", "mean"))
        .reset_index())

fig2, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = sns.color_palette("Set2", len(dept))
axes[0].barh(dept["department"], dept["avg_revenue"], color=palette)
axes[0].set_title("Avg Revenue per Employee by Dept.")
axes[0].set_xlabel("Avg Revenue (£)")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
axes[1].barh(dept["department"], dept["avg_margin"], color=palette)
axes[1].set_title("Avg Profit Margin % by Dept.")
axes[1].set_xlabel("Margin %")
plt.tight_layout(); plt.show()

## 19 · Inventory Risk

In [29]:
inv_pdf = ea.inventory_risk().toPandas()
for c in ["days_of_cover", "daily_demand", "stock"]:
    inv_pdf[c] = pd.to_numeric(inv_pdf[c], errors="coerce").astype(float)

color_map = {"STOCKOUT RISK": "#e74c3c", "OVERSTOCKED": "#f39c12", "OK": "#2ecc71"}
status_counts = inv_pdf["inventory_status"].value_counts().reset_index()
status_counts.columns = ["status", "count"]

fig = px.pie(status_counts, values="count", names="status", color="status",
             color_discrete_map=color_map,
             title="Inventory Status Distribution", hole=0.4)
fig.update_traces(textinfo="percent+label+value")
fig.show()


━━━ 12. Inventory Risk ━━━


+----------+------------+-----------+-----+----------+-----------+----------+------------+-------------+----------------+
|product_id|product_name|category   |stock|total_sold|order_count|sales_days|daily_demand|days_of_cover|inventory_status|
+----------+------------+-----------+-----+----------+-----------+----------+------------+-------------+----------------+
|34        |Feeling     |Sports     |26   |7         |1          |0         |7.0         |4.0          |STOCKOUT RISK   |
|318       |Reason      |Apparel    |22   |6         |1          |0         |6.0         |4.0          |STOCKOUT RISK   |
|363       |Within      |Electronics|46   |10        |1          |0         |10.0        |5.0          |STOCKOUT RISK   |
|983       |Capital     |Furniture  |27   |6         |1          |0         |6.0         |5.0          |STOCKOUT RISK   |
|997       |Star        |Electronics|37   |7         |1          |0         |7.0         |5.0          |STOCKOUT RISK   |
|785       |Upon        

In [30]:
risky = inv_pdf[inv_pdf["inventory_status"] != "OK"].copy()
fig2 = px.scatter(risky, x="daily_demand", y="stock",
                  color="inventory_status", hover_name="product_name",
                  hover_data=["category", "days_of_cover"],
                  title="Risky Products — Daily Demand vs Current Stock",
                  labels={"daily_demand": "Daily Demand (units)", "stock": "Stock",
                          "inventory_status": "Status"},
                  color_discrete_map=color_map, size="days_of_cover", size_max=25)
fig2.show()

print("\n🚨 Top 10 Stockout Risk Products:")
display(inv_pdf[inv_pdf["inventory_status"] == "STOCKOUT RISK"]
        .sort_values("days_of_cover").head(10)
        [["product_name", "category", "stock", "daily_demand", "days_of_cover"]]
        .reset_index(drop=True))


🚨 Top 10 Stockout Risk Products:


,product_name,category,stock,daily_demand,days_of_cover
0,Feeling,Sports,26.0,7.0,4.0
1,Reason,Apparel,22.0,6.0,4.0
2,Within,Electronics,46.0,10.0,5.0
3,Capital,Furniture,27.0,6.0,5.0
4,Star,Electronics,37.0,7.0,5.0
5,Upon,Furniture,35.0,7.0,5.0
6,Community,Furniture,49.0,10.0,5.0
7,Require,Apparel,20.0,4.0,5.0
8,Mind,Books,57.0,9.0,6.0
9,Here,Apparel,22.0,4.0,6.0


## 20 · Month-over-Month Growth

In [31]:
ts_pdf = sales["time_series_analysis"].toPandas()
for c in ["monthly_revenue", "mom_growth_pct"]:
    ts_pdf[c] = pd.to_numeric(ts_pdf[c], errors="coerce").astype(float)
ts_pdf["period"] = (ts_pdf["year"].astype(str) + "-" +
                    ts_pdf["month"].astype(str).str.zfill(2))

fig = px.bar(ts_pdf, x="period", y="mom_growth_pct", color="growth_category",
             facet_col="category", facet_col_wrap=3,
             title="Month-over-Month Growth % by Category",
             labels={"mom_growth_pct": "MoM Growth %", "period": "Month"},
             color_discrete_map={"High Growth": "#2ecc71", "Growing": "#3498db",
                                  "Stable": "#f39c12", "Declining": "#e74c3c"})
fig.update_layout(height=700, xaxis_tickangle=-45)
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5)
fig.show()

## 21 · Advanced Window Analytics

In [32]:
ma_sorted = ma_pdf.sort_values("sale_date")

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=False)
axes[0].fill_between(ma_sorted["sale_date"], ma_sorted["daily_revenue"],
                     alpha=0.25, color="steelblue")
axes[0].plot(ma_sorted["sale_date"], ma_sorted["daily_revenue"],
             color="steelblue", linewidth=0.8, label="Daily Revenue")
axes[0].plot(ma_sorted["sale_date"], ma_sorted["moving_avg_7days"],
             color="blue", linewidth=1.8, label="7-Day MA")
axes[0].plot(ma_sorted["sale_date"], ma_sorted["moving_avg_30days"],
             color="darkred", linewidth=2.2, label="30-Day MA")
axes[0].set_title("Daily Revenue with Moving Averages")
axes[0].set_ylabel("Revenue (£)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
axes[0].legend(fontsize=9)

axes[1].hist(ma_sorted["daily_revenue"].dropna(), bins=40,
             color="coral", edgecolor="white", alpha=0.85)
mean_rev = float(ma_sorted["daily_revenue"].mean())
axes[1].axvline(mean_rev, color="darkred", linestyle="--",
                label=f"Mean £{mean_rev:,.0f}")
axes[1].set_title("Daily Revenue Distribution")
axes[1].set_xlabel("Revenue (£)"); axes[1].set_ylabel("Days")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

## 22 · Executive KPI Summary

In [33]:
kpi = spark.sql('''
    SELECT
        COUNT(DISTINCT customer_id)                               AS total_customers,
        ROUND(SUM(lifetime_value), 0)                             AS total_ltv,
        ROUND(AVG(lifetime_value), 2)                             AS avg_ltv,
        ROUND(AVG(total_orders), 2)                               AS avg_orders,
        SUM(CASE WHEN churn_risk = "High" THEN 1 ELSE 0 END)      AS high_churn
    FROM customer_analytics
''').toPandas()

prod_kpi = spark.sql('''
    SELECT COUNT(*) AS total_products, ROUND(AVG(profit_margin), 2) AS avg_margin
    FROM product_performance
''').toPandas()

store_kpi = spark.sql('''
    SELECT COUNT(*) AS total_stores, ROUND(AVG(profit_margin), 2) AS avg_store_margin
    FROM store_performance
''').toPandas()

print("=" * 52)
print("  EXECUTIVE KPI DASHBOARD")
print("=" * 52)
print(f"  Total Customers        : {int(kpi['total_customers'][0]):>10,}")
print(f"  Total Revenue (LTV)    : £{float(kpi['total_ltv'][0]):>12,.0f}")
print(f"  Avg Customer LTV       : £{float(kpi['avg_ltv'][0]):>12,.2f}")
print(f"  Avg Orders/Customer    : {float(kpi['avg_orders'][0]):>10.2f}")
print(f"  High Churn Risk        : {int(kpi['high_churn'][0]):>10,}")
print(f"  Total Products         : {int(prod_kpi['total_products'][0]):>10,}")
print(f"  Avg Product Margin     : {float(prod_kpi['avg_margin'][0]):>9.2f}%")
print(f"  Total Stores           : {int(store_kpi['total_stores'][0]):>10,}")
print(f"  Avg Store Margin       : {float(store_kpi['avg_store_margin'][0]):>9.2f}%")
print("=" * 52)

  EXECUTIVE KPI DASHBOARD
  Total Customers        :        485
  Total Revenue (LTV)    : £   2,272,589
  Avg Customer LTV       : £    4,685.75
  Avg Orders/Customer    :       2.06
  High Churn Risk        :        366
  Total Products         :        616
  Avg Product Margin     :     51.32%
  Total Stores           :        485
  Avg Store Margin       :     51.73%


## 23 · Cleanup

In [34]:
spark.stop()
print("Spark session stopped. All done ✅")

Spark session stopped. All done ✅
